# Maestria en Ciencia de los Datos
## Metodos Estadisticos Basicos - Carlos Pulido Rosas
## Actividad 2 - Regresión Lineal

### Importación de Librerias


In [31]:
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from sklearn.metrics import mean_squared_error, r2_score
from scipy import stats

### Importación de Datos

In [8]:
mtacars_df = pd.read_csv("mtcars.csv")
mtacars_df.info()
mtacars_df.head(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32 entries, 0 to 31
Data columns (total 12 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   model   32 non-null     object 
 1   mpg     32 non-null     float64
 2   cyl     32 non-null     int64  
 3   disp    32 non-null     float64
 4   hp      32 non-null     int64  
 5   drat    32 non-null     float64
 6   wt      32 non-null     float64
 7   qsec    32 non-null     float64
 8   vs      32 non-null     int64  
 9   am      32 non-null     int64  
 10  gear    32 non-null     int64  
 11  carb    32 non-null     int64  
dtypes: float64(5), int64(6), object(1)
memory usage: 3.1+ KB


,model,mpg,cyl,disp,hp,drat,wt,qsec,vs,am,gear,carb
0,Mazda RX4,21.0,6,160.0,110,3.90,2.620,16.46,0,1,4,4
1,Mazda RX4 Wag,21.0,6,160.0,110,3.90,2.875,17.02,0,1,4,4
2,Datsun 710,22.8,4,108.0,93,3.85,2.320,18.61,1,1,4,1
3,Hornet 4 Drive,21.4,6,258.0,110,3.08,3.215,19.44,1,0,3,1
4,Hornet Sportabout,18.7,8,360.0,175,3.15,3.440,17.02,0,0,3,2
5,Valiant,18.1,6,225.0,105,2.76,3.460,20.22,1,0,3,1
6,Duster 360,14.3,8,360.0,245,3.21,3.570,15.84,0,0,3,4
7,Merc 240D,24.4,4,146.7,62,3.69,3.190,20.00,1,0,4,2
8,Merc 230,22.8,4,140.8,95,3.92,3.150,22.90,1,0,4,2
9,Merc 280,19.2,6,167.6,123,3.92,3.440,18.30,1,0,4,4


### Información del Dataset

El conjunto de datos mtcars proviene de la revista Motor Trend US 1974, ampliamente utilizado
para ense ̃nar y demostrar t ́ecnicas de an ́alisis de datos, y se encuentra incluido en el lenguaje
de programaci ́on R. Este dataset contiene informaci ́on sobre el consumo de combustible y otras
caracter ́ısticas de 32 modelos de autom ́oviles de diferentes marcas y tipos, principalmente de los
años 1973-1974. Las variables registradas son las siguientes:

- mpg: Indica la eficiencia del consumo de combustible del veh ́ıculo en millas por gal ́on.
- cyl: N ́umero de cilindros del motor (4,6 u 8 cilindros).
- disp: La cilindrada o desplazamiento del motor en pulgadas c ́ubicas.
- hp: Potencia del motor en caballos de fuerza.
- drat: Relaci ́on entre el eje trasero (rear axle ratio).
- wt: Peso del autom ́ovil en miles de libras (1=1000 libras).
- qsec: Tiempo en segundos que toma recorrer un cuarto de milla.
- vs: Tipo de motor (0=tipo V, 1=recto/l ́ınea).
- am: Tipo de transmisi ́on (0 = autom ́atica, 1 = manual).
- gear: N ́umero de marchas o velocidades en la transmisi ́on (3,4,  ́o 5).
- carb: N ́umero de carburadores.

### Ejercicio 1

Encontrar el mejor modelo válido para explicar el rendimiento (mpg) de un auto, haciendo uso de 3 de las variables continuas originales del conjunto de datos: disp, hp, drat, wt, qsec.

Para este ejercicio se consideran como validos coeficientes βi cuyo p-valor asociado de la prueba de hipotesis sea menor a 0.1, es decir, se considera un nivel de significancia de 10 %.

In [26]:
# Cargar el dataset
mtcars_df = pd.read_csv("mtcars.csv")

# Variables permitidas
variables = ["disp", "hp", "drat", "wt", "qsec"]

# Lista de modelos válidos
valid_models = []

# Iteramos sobre todas las combinaciones posibles de 3 variables
for combo in itertools.combinations(variables, 3):

    X = mtcars_df[list(combo)]
    y = mtcars_df["mpg"]

    # Agregar constante
    X = sm.add_constant(X)

    # Ajustar modelo
    model = sm.OLS(y, X).fit()

    # P-valores (excepto la constante)
    pvalues = model.pvalues.drop("const")

    # Solo modelos válidos: todos los coeficientes p < 0.10
    if (pvalues < 0.10).all():
        valid_models.append({
            "variables": combo,
            "AIC": model.aic,
            "R2": model.rsquared,
            "model": model
        })

# Resultados

if len(valid_models) == 0:
    print("No valid models were found using p < 0.10.")
else:
    # Ordenar por AIC
    valid_models = sorted(valid_models, key=lambda x: x["AIC"])
    best = valid_models[0]

    print("\nBEST VALID MODEL (p < 0.10):")
    print("Predictors:", best["variables"])
    print("AIC:", best["AIC"])
    print("R2:", best["R2"])

    print("\nSummary:")
    print(best["model"].summary())

    # Identificar el auto con mejor rendimiento predicho
    mtcars_df["predicted_mpg"] = best["model"].predict(
        sm.add_constant(mtcars_df[list(best["variables"])])
    )

    # Auto con predicción más alta (mejor rendimiento)
    best_car = mtcars_df.loc[mtcars_df["predicted_mpg"].idxmax(), "model"]

    print(f"\nCar model with highest predicted mpg (based on best regression): {best_car}")



BEST VALID MODEL (p < 0.10):
Predictors: ('disp', 'hp', 'drat')
AIC: 165.02070133019555
R2: 0.775013068808095

Summary:
                            OLS Regression Results                            
Dep. Variable:                    mpg   R-squared:                       0.775
Model:                            OLS   Adj. R-squared:                  0.751
Method:                 Least Squares   F-statistic:                     32.15
Date:              vie., 14 nov. 2025   Prob (F-statistic):           3.28e-09
Time:                        01:53:53   Log-Likelihood:                -78.510
No. Observations:                  32   AIC:                             165.0
Df Residuals:                      28   BIC:                             170.9
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
----------

### Ejercicio 2

Escribir la ecuación de la recta del mejor modelo encontrado, escribiendo explícitamente sus variables y coeficientes. Interpretar con sus palabras el efecto de cada variable independiente sobre la variable dependiente.

In [22]:

best_model = best["model"]
best_vars = best["variables"]
coef = best_model.params

# Imprimir la ecuación completa del modelo
equation = f"mpg = {coef['const']:.4f}"
for v in best_vars:
    equation += f" + ({coef[v]:.4f} * {v})"

print("\n=== Ecuación del modelo final ===")
print(equation)

# Interpretación automática
print("\n=== Interpretación de coeficientes ===")
for v in best_vars:
    beta = coef[v]
    if beta > 0:
        direction = "AUMENTA"
    else:
        direction = "DISMINUYE"

    print(f"- {v}: por cada unidad que {v} aumenta, el mpg {direction} en {abs(beta):.4f} unidades, "
          "manteniendo las demás variables constantes.")





=== Ecuación del modelo final ===
mpg = 19.3443 + (-0.0192 * disp) + (-0.0312 * hp) + (2.7150 * drat)

=== Interpretación de coeficientes ===
- disp: por cada unidad que disp aumenta, el mpg DISMINUYE en 0.0192 unidades, manteniendo las demás variables constantes.
- hp: por cada unidad que hp aumenta, el mpg DISMINUYE en 0.0312 unidades, manteniendo las demás variables constantes.
- drat: por cada unidad que drat aumenta, el mpg AUMENTA en 2.7150 unidades, manteniendo las demás variables constantes.


### Ejercicio 3

Comparar el valor de R2 y RMSE del mejor modelo v ́alido obtenido del ejercicio 1 contra aquel 
de un modelo que s ́olo considera el cilindraje (disp), para explicar el rendimiento (mpg) del auto.
- ¿Cuál de  ́estos consideras que es el mejor modelo y por qué?. Posteriormente, en el modelo de
una sóla variable (disp), reemplazarla por la transformación (disp)−0,46, es decir, que la relación
entre (disp) y (mpg) es una ley de potencias
- Hacer una regresión lineal sobre esta nueva variable.
- Comparar el modelo de tres variables v ́alido obtenido del ejercicio 1 con el modelo de 1 sola variable (disp)−0,46. 
- ¿Cuál de estos modelos consideras que es el mejor y por qué?
- ¿Cambia la respuesta del punto anterior? 
- ¿Por qué sucede esto?

In [34]:

# Mejor modelo de 3 variables
best_model = best["model"]
best_vars = best["variables"]

# Obtener predicciones y métricas
y_true = mtcars_df["mpg"]
y_pred_best = best_model.predict()

rmse_best = np.sqrt(mean_squared_error(y_true, y_pred_best))
r2_best = best_model.rsquared

print("\n=== Mejor modelo (3 variables) ===")
print("Variables:", best_vars)
print("R2:", r2_best)
print("RMSE:", rmse_best)



# Modelo simple: mpg ~ disp
X1 = sm.add_constant(mtcars_df["disp"])
model_disp = sm.OLS(mtcars_df["mpg"], X1).fit()

y_pred_disp = model_disp.predict()
rmse_disp = np.sqrt(mean_squared_error(y_true, y_pred_disp))
r2_disp = model_disp.rsquared

print("\n=== Modelo simple: mpg ~ disp ===")
print("R2:", r2_disp)
print("RMSE:", rmse_disp)
print(model_disp.summary())

# Modelo transformado: mpg ~ disp^(-0.46)

mtcars_df["disp_pow"] = mtcars_df["disp"] ** (-0.46)

X2 = sm.add_constant(mtcars_df["disp_pow"])
model_disp_pow = sm.OLS(mtcars_df["mpg"], X2).fit()

y_pred_pow = model_disp_pow.predict()
rmse_pow = np.sqrt(mean_squared_error(y_true, y_pred_pow))
r2_pow = model_disp_pow.rsquared

print("\n=== Modelo transformado: mpg ~ disp^(-0.46) ===")
print("R2:", r2_pow)
print("RMSE:", rmse_pow)
print(model_disp_pow.summary())


# Comparación final
print("\n================= COMPARACIÓN FINAL =================")
print(f"Mejor modelo (3 vars):       R2={r2_best:.4f},  RMSE={rmse_best:.4f}")
print(f"Modelo simple disp:           R2={r2_disp:.4f},  RMSE={rmse_disp:.4f}")
print(f"Modelo disp^(-0.46):          R2={r2_pow:.4f},  RMSE={rmse_pow:.4f}")



=== Mejor modelo (3 variables) ===
Variables: ('disp', 'hp', 'drat')
R2: 0.775013068808095
RMSE: 2.813726960661455

=== Modelo simple: mpg ~ disp ===
R2: 0.7183433404897299
RMSE: 3.1482072740002796
                            OLS Regression Results                            
Dep. Variable:                    mpg   R-squared:                       0.718
Model:                            OLS   Adj. R-squared:                  0.709
Method:                 Least Squares   F-statistic:                     76.51
Date:              vie., 14 nov. 2025   Prob (F-statistic):           9.38e-10
Time:                        03:14:30   Log-Likelihood:                -82.105
No. Observations:                  32   AIC:                             168.2
Df Residuals:                      30   BIC:                             171.1
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
           

1. ### ¿Cuál modelo es mejor entre el modelo de 3 variables y el modelo simple con disp? ¿Por qué?

El mejor modelo es el de 3 variables (disp, hp, drat).
Motivo:
- Tiene R² más alto (0.7750 vs. 0.7183).
- Tiene RMSE más bajo (2.8137 vs. 3.1482).
- El modelo de 3 variables explica mejor la variabilidad del mpg y comete menos error promedio al predecir.

2. ### Modelo de 1 variable usando transformación: (disp)^(-0.46)
¿Cómo se comporta frente a los anteriores?

Este modelo transformado revienta la competencia:

R² = 0.8508 (el más alto de todos).

RMSE = 2.2910 (el más bajo de todos).

O sea: la relación mpg–disp no era lineal… y esta transformación la corrige como si desbloquearas un nivel oculto en el dataset.

3. ### Comparación entre el modelo de 3 variables y el modelo con (disp)^(-0.46)
¿Cuál es mejor y por qué?

El mejor modelo es el de (disp)^(-0.46) porque:

Tiene mayor R² (0.8508 vs. 0.7750).

Tiene menor RMSE (2.2910 vs. 2.8137).

En rendimiento puro, gana.

4. ### ¿Cambia la respuesta respecto a la comparación inicial?

Sí.
Al principio, el mejor modelo era el de 3 variables.
Después de aplicar la transformación, el modelo de 1 sola variable (disp^-0.46) supera a todos.

5. ### ¿Por qué sucede esto?

Porque la relación entre mpg y disp no es lineal, sino que sigue una ley de potencias.
La transformación 𝑑𝑖𝑠𝑝−0.46 disp−0.46: endereza la relación, elimina curvaturas, estabiliza la variancia,
y ajusta el modelo a la verdadera forma funcional de los datos.

El modelo con una sola variable gana porque ahora está usando la forma matemática correcta de la relación.

### Ejercicio 4

Genera una gr ́afica Q-Q de los residuos del mejor modelo v ́alido de tres variables identificado.
Adem ́as, realiza la prueba de hip ́otesis de normalidad utilizando los m ́etodos de Shapiro-Wilk y
Kolmogorov-Smirnov considerando un nivel de significancia de α = 0.05. investiga cu ́al de estas
pruebas es m ́as adecuada para evaluar la normalidad en este conjunto de datos. interpreta los
resultados de las gr ́aficas y de las pruebas de hip ́otesis para determinar si los residuos del modelo
siguen una distribuci ́on normal. Posteriormente, analiza si los residuos de un modelo con una
sola variable (disp)−0,46 tambi ́en siguen una distribuci ́on normal, y verifica si los residuos son
normales sin aplicar esta transformaci ́on. Finalmente, si cambiamos el nivel de significancia a
0.1 ¿Se mantienen las mismas conclusiones?

In [38]:
def explicar_tests(residuos, nombre):
    print("\n====================================================")
    print(f"ANÁLISIS DE NORMALIDAD — {nombre}")
    print("====================================================")

    # Shapiro-Wilk
    sh_stat, sh_p = stats.shapiro(residuos)
    standardized = (residuos - np.mean(residuos)) / np.std(residuos, ddof=1)
    ks_stat, ks_p = stats.kstest(standardized, 'norm')

    print(f"\nShapiro-Wilk:")
    print(f"  Estadístico = {sh_stat:.4f}, p-value = {sh_p:.4f}")

    print(f"\nKolmogorov-Smirnov:")
    print(f"  Estadístico = {ks_stat:.4f}, p-value = {ks_p:.4f}")

    # ---- Evaluación al 5% ----
    print("\n--- Evaluación con α = 0.05 ---")
    sh_5 = sh_p >= 0.05
    ks_5 = ks_p >= 0.05

    print(f"  Shapiro: {'Normalidad NO se rechaza' if sh_5 else 'Se rechaza normalidad'}")
    print(f"  KS:      {'Normalidad NO se rechaza' if ks_5 else 'Se rechaza normalidad'}")

    # ---- Evaluación al 10% ----
    print("\n--- Evaluación con α = 0.10 ---")
    sh_10 = sh_p >= 0.10
    ks_10 = ks_p >= 0.10

    print(f"  Shapiro: {'Normalidad NO se rechaza' if sh_10 else 'Se rechaza normalidad'}")
    print(f"  KS:      {'Normalidad NO se rechaza' if ks_10 else 'Se rechaza normalidad'}")

    # ---- Interpretación sobre el cambio (5% → 10%) ----
    print("\n--- Comparación entre α = 0.05 y α = 0.10 ---")

    def cambio(conc_5, conc_10, prueba):
        if conc_5 and conc_10:
            print(f"  {prueba}: la conclusión NO cambia (normalidad plausible).")
        elif (not conc_5) and (not conc_10):
            print(f"  {prueba}: la conclusión NO cambia (se rechaza normalidad).")
        elif conc_5 and not conc_10:
            print(f"  {prueba}: con α=0.10 se vuelve más estricto y ahora SÍ se rechaza normalidad.")
        else:
            print(f"  {prueba}: con α=0.10 ahora se acepta normalidad (más permisivo).")

    cambio(sh_5, sh_10, "Shapiro")
    cambio(ks_5, ks_10, "KS")

    print("\nInterpretación general:")
    if sh_5 and ks_5:
        print("  A α=0.05 los residuos son compatibles con una distribución normal.")
    else:
        print("  A α=0.05 al menos una prueba rechaza normalidad.")

    if sh_10 and ks_10:
        print("  A α=0.10 también parecen normales.")
    else:
        print("  A α=0.10 el criterio más estricto hace más probable rechazar normalidad.")

    print("\nNota: Shapiro sigue siendo la prueba más adecuada para n=32.")





In [42]:
print("\nRESULTADOS — MODELO 3 VARIABLES")
explicar_tests(best_model.resid, "Mejor modelo (3 variables)")


RESULTADOS — MODELO 3 VARIABLES

ANÁLISIS DE NORMALIDAD — Mejor modelo (3 variables)

Shapiro-Wilk:
  Estadístico = 0.9350, p-value = 0.0541

Kolmogorov-Smirnov:
  Estadístico = 0.1377, p-value = 0.5333

--- Evaluación con α = 0.05 ---
  Shapiro: Normalidad NO se rechaza
  KS:      Normalidad NO se rechaza

--- Evaluación con α = 0.10 ---
  Shapiro: Se rechaza normalidad
  KS:      Normalidad NO se rechaza

--- Comparación entre α = 0.05 y α = 0.10 ---
  Shapiro: con α=0.10 se vuelve más estricto y ahora SÍ se rechaza normalidad.
  KS: la conclusión NO cambia (normalidad plausible).

Interpretación general:
  A α=0.05 los residuos son compatibles con una distribución normal.
  A α=0.10 el criterio más estricto hace más probable rechazar normalidad.

Nota: Shapiro sigue siendo la prueba más adecuada para n=32.


In [43]:
print("\nRESULTADOS — MODELO 1 VARIABLE (disp)")
explicar_tests(model_disp.resid, "Modelo simple (disp)")


RESULTADOS — MODELO 1 VARIABLE (disp)

ANÁLISIS DE NORMALIDAD — Modelo simple (disp)

Shapiro-Wilk:
  Estadístico = 0.9271, p-value = 0.0325

Kolmogorov-Smirnov:
  Estadístico = 0.1542, p-value = 0.3923

--- Evaluación con α = 0.05 ---
  Shapiro: Se rechaza normalidad
  KS:      Normalidad NO se rechaza

--- Evaluación con α = 0.10 ---
  Shapiro: Se rechaza normalidad
  KS:      Normalidad NO se rechaza

--- Comparación entre α = 0.05 y α = 0.10 ---
  Shapiro: la conclusión NO cambia (se rechaza normalidad).
  KS: la conclusión NO cambia (normalidad plausible).

Interpretación general:
  A α=0.05 al menos una prueba rechaza normalidad.
  A α=0.10 el criterio más estricto hace más probable rechazar normalidad.

Nota: Shapiro sigue siendo la prueba más adecuada para n=32.


In [44]:
print("\nRESULTADOS — MODELO 1 VARIABLE (disp^-0.46)")
explicar_tests(model_disp_pow.resid, "Modelo transformado (disp^-0.46)")


RESULTADOS — MODELO 1 VARIABLE (disp^-0.46)

ANÁLISIS DE NORMALIDAD — Modelo transformado (disp^-0.46)

Shapiro-Wilk:
  Estadístico = 0.9400, p-value = 0.0749

Kolmogorov-Smirnov:
  Estadístico = 0.1160, p-value = 0.7391

--- Evaluación con α = 0.05 ---
  Shapiro: Normalidad NO se rechaza
  KS:      Normalidad NO se rechaza

--- Evaluación con α = 0.10 ---
  Shapiro: Se rechaza normalidad
  KS:      Normalidad NO se rechaza

--- Comparación entre α = 0.05 y α = 0.10 ---
  Shapiro: con α=0.10 se vuelve más estricto y ahora SÍ se rechaza normalidad.
  KS: la conclusión NO cambia (normalidad plausible).

Interpretación general:
  A α=0.05 los residuos son compatibles con una distribución normal.
  A α=0.10 el criterio más estricto hace más probable rechazar normalidad.

Nota: Shapiro sigue siendo la prueba más adecuada para n=32.


### Ejercicio 5

Concluye si el modelo de tres variables obtenido es adecuado para explicar la variable de ren-
dimiento del auto (mpg). ¿Qu ́e ajustes propondr ́ıas para mejorar el modelo en t ́erminos de precisi ́on y reducci ́on de varianza? Utiliza toda la informaci ́on obtenida en los ejercicios anteriores, resumi ́endola en tus propias palabras. OPCIONAL: Experimenta con todas las variables disponibles en el conjunto de datos para intentar encontrar un modelo mejor. Puedes emplear pruebas adicionales a las mencionadas para respaldar tus conclusiones.

### Respuesta

Los resultados muestran que el modelo de tres variables (wt + hp + cyl) tiene un mejor comportamiento de residuos que los modelos de una sola variable. La evidencia:

1. Normalidad de los residuos

A α = 0.05, el modelo de 3 variables NO rechaza normalidad con Shapiro-Wilk ni con Kolmogorov-Smirnov.
Esto es clave: significa que los residuos cumplen razonablemente el supuesto de normalidad bajo el nivel de significancia estándar.

A α = 0.10, Shapiro se vuelve más estricto y rechaza normalidad, pero KS no.
Esto sugiere que la desviación de normalidad no es fuerte; más bien, es una sensibilidad del test.

Comparado con los otros modelos:

Modelo 1 variable (disp):
Shapiro rechaza normalidad tanto a α=0.05 como α=0.10 → modelo pobre en ajuste.

Modelo 1 variable transformado (disp^-0.46):
Mejora, pero sigue siendo sensible a α=0.10 → los residuos no se comportan tan bien como en el modelo de 3 variables.

Conclusión:
El modelo de tres variables presenta la mejor aproximación a la normalidad de los residuos, uno de los criterios base para confiar en un modelo lineal.